# How much brighter is the real sky than Kelsall, and where? The offset map (Nov 8, 2026)

### Summary
* Dr. Saydjari's Week 13 question: does the model need a term for the mean (static) sky, an offset as a function of
  Galactic l, b? Answered with real SPHEREx data on **48 patches across the sky** (0.92° each, one per level-1 HEALPix
  centre) at **8 wavelengths, 1.3–5 μm**.
* **The tall-table web service is still down**, so the database partitions were downloaded from Flatiron's public file
  server to Discovery scratch (48 files, 84.5 GB, ~15 min) and queried locally with DuckDB, using the same SQL as before.
* **At 5 μm, away from the Galactic plane, the offset is zodi that Kelsall misses, not a fixed sky:**
  offset = 0.033 + **0.19 × Kelsall** MJy/sr (R² 0.74; correlation with |β| −0.85, with csc|b| +0.03).
* **At every wavelength from 1.6 to 5 μm, the real zodi is ~11–27% brighter than Kelsall.** Below ~3 μm there is also a
  Galactic term (∝ csc|b|, i.e. starlight; no star masking yet). The per-patch fits over time agree: the time-variable
  part is 5–31% larger than Kelsall's.
* **Against SPHEREx's own zodi model there is no proportional part** at any wavelength, only a constant 0.025–0.055 MJy/sr.
  So the SPHEREx pipeline has already absorbed this correction.
* **Answer for the model:** a fixed l, b map is not the main thing missing. The main thing is a **~10–25% scale-up of the
  zodi**, plus a small constant and, at short wavelengths, a Galactic term.
* For the mentor's "replace the ZodiPy column with ours": every patch now has a per-exposure file with our model next to
  the measured flux, SPHEREx's zodi and ZodiPy (§4).


In [ ]:
const R = joinpath(@__DIR__, "..")
struct PNGFile; path::String; end; Base.show(io::IO, ::MIME"image/png", p::PNGFile) = write(io, read(p.path))   # raw PNG bytes: renders in any Jupyter front end
showpng(f) = PNGFile(joinpath(R, "figures", f))
using DelimitedFiles, Printf
D = joinpath(R, "data", "spherex_real")     # git-ignored; made by realdata/talltable_offsetmap.jl and offset_fit.jl


## 1. Getting the data without the web service
The tall table's web service has returned "database has been invalidated" since Nov 4. Its documentation says the
partitions (one Parquet file per adaptively refined HEALPix cell) can also be downloaded directly, along with
`parts.txt`, `image.parquet` and `waves.parquet`. With `TALLTABLE_DB_DIR` pointed at the copy, the client's
`PixelQuery(web=False)` writes the same SQL against local files.

* `realdata/talltable_local_size.py` lists the partitions a grid needs and their sizes: grid 1 (48 patches) = 48
  partitions, 84.5 GB; grid 2 (192 patches) = 228 partitions, 389 GB. The north-pole patch at this size alone would be 64
  partitions, 150 GB (the deep field), so it was left out; it was already measured at level 8.
* `pipeline/cluster/talltable_download.sbatch G [windows]` downloads at ~90 MB/s on a compute node, then runs
  `talltable_bins.py` with `TALLTABLE_LOCAL=1`: ~2 s per patch per wavelength window.
* One fix along the way: each grid centre sits exactly on a corner of 4 level-6 pixels, so the rounding of the
  coordinates decides which pixel is used. The sizing script now rounds exactly as the binning script does.
* Per patch: 7–67 exposures (median ~38), 2–5 million pixels in 4.9–5.1 μm. The default flag filter is the same as
  before; no star masking.


## 2. The offset map at 4.9–5.1 μm
Per patch, offset = median over exposures of (measured − our model). Our model = the 30 networks of rounds 37–38, moved to
each exposure's wavelength with ZodiPy's spectral shape. On these exposures it equals ZodiPy to ≤ 0.036%.


In [ ]:
showpng("11082026_offsetmap_L6_grid1.png")


* **Near the Galactic plane (|b| < 10°)** the offset jumps to 0.2–1.45 MJy/sr. The largest is at l = 10°, b = 2°, next to
  the Galactic centre. That is the Galaxy (stars and dust emission, no masking).
* **Everywhere else** it is 0.075–0.19 MJy/sr, and it is largest where the zodi is brightest (near the ecliptic).

Fits over the 34 patches with |b| > 15°:

| model of the offset | coefficients (MJy/sr) | rms left | R² |
|---|---|---|---|
| c0 + c1·Kelsall | c0 = 0.033 ± 0.010, **c1 = 0.188 ± 0.020** | 0.015 | 0.74 |
| c0 + c1·Kelsall + c2·csc\|b\| | 0.017, 0.195, c2 = 0.0075 ± 0.0036 | 0.014 | 0.77 |
| c0 + c2·csc\|b\| (a Galactic sky only) | 0.127, 0.001 | 0.030 | **0.00** |
| SPHEREx zodi model: c0 + c1·(its model) | c0 = 0.043 ± 0.008, c1 = −0.005 ± 0.014 | 0.011 | 0.00 |

So at 5 μm the offset is about **19% of Kelsall** plus a ~0.03 MJy/sr constant. A Galactic (csc|b|) sky explains none of
it. SPHEREx's zodi model leaves only a constant, about the size expected for the extragalactic background plus faint stars.


## 3. The same at 8 wavelengths
The same partitions, queried in 0.1 μm windows (`TT_LAM=lo:hi`); ZodiPy and our model at each exposure's wavelength.
ZodiPy's DIRBE model starts at 1.25 μm, so the windows start there. |b| > 15°, 34 patches:

| window (μm) | Kelsall, median (MJy/sr) | c1, Kelsall only | c1 with a csc\|b\| term | c2 (csc\|b\|) | R² with c2 | per-patch time-fit scale a | SPHEREx: c0, c1 |
|---|---|---|---|---|---|---|---|
| 1.25–1.35 | 0.184 | 0.03 ± 0.04 | 0.05 ± 0.02 | 0.015 | 0.54 | 1.21 | 0.054, −0.07 |
| 1.6–1.7 | 0.162 | 0.10 ± 0.03 | 0.13 ± 0.02 | 0.014 | 0.76 | 1.31 | 0.043, −0.05 |
| 2.2–2.3 | 0.111 | 0.11 ± 0.04 | 0.14 ± 0.02 | 0.009 | 0.73 | 1.13 | 0.029, −0.04 |
| 2.8–2.9 | 0.088 | 0.09 ± 0.03 | 0.11 ± 0.02 | 0.007 | 0.67 | 1.14 | 0.027, −0.04 |
| 3.4–3.5 | 0.077 | 0.25 ± 0.04 | 0.27 ± 0.03 | 0.008 | 0.77 | 1.21 | 0.026, −0.05 |
| 4.05–4.15 | 0.143 | 0.17 ± 0.03 | 0.18 ± 0.03 | 0.007 | 0.64 | 1.09 | 0.032, −0.03 |
| 4.5–4.6 | 0.289 | 0.13 ± 0.02 | 0.14 ± 0.02 | 0.006 | 0.58 | 1.05 | 0.043, −0.02 |
| 4.9–5.1 | 0.475 | 0.19 ± 0.02 | 0.19 ± 0.02 | 0.008 | 0.77 | 1.09 | 0.043, −0.00 |


In [ ]:
showpng("11082026_offset_spectrum_L6_grid1.png")


In [ ]:
A, h = readdlm(joinpath(D, "ttbin_L6_grid1_offset_fit.csv"), ',', header=true)
println(join(h, "  ")); for i in axes(A, 1); println(join((x isa AbstractFloat ? @sprintf("%.4g", x) : string(x) for x in A[i, :]), "  ")); end


* **Kelsall is too faint at every wavelength from 1.6 μm up:** by 11–14% in the scattered-light range (1.6–2.9 μm) and
  14–27% in the thermal range (3.4–5 μm). Two independent measures agree on the sign: across the sky (c1) and over time
  within each patch (a = 1.05–1.31).
* **The 1.3 μm window is the exception** (c1 ≈ 0.05). The constant and Galactic terms are largest there, so the zodi term
  is hardest to separate.
* **The Galactic term grows toward short wavelengths** (c2 0.006 → 0.015 MJy/sr). Including it raises R² from 0.2 to
  ~0.75 below 3 μm. This is what unmasked starlight does, so star masking (when the user says so) should shrink it.
* **The 3.45 μm point is the highest** (c1 0.27). DIRBE has a band at 3.5 μm, so this is not ZodiPy interpolating between
  bands. Whether it is real or one window's noise needs the finer grid (overnight run, §5).
* **SPHEREx's zodi model has c1 ≈ 0 (−0.07 to 0.00) everywhere**, so its pipeline zodi already includes this scale-up. Its
  leftover constant (0.025–0.055 MJy/sr, lowest at 2–3.5 μm) is the sky that is not zodi.


In [ ]:
showpng("11082026_offsetmap_L6_grid1_w1.6.png")


In [ ]:
showpng("11082026_offsetmap_L6_grid1_w3.4.png")


## 4. Our model in place of the ZodiPy column
For every patch and window, `realdata/talltable_offsetmap.jl` now writes `ttbin_L6_<patch><window>_ours.csv`: one row per
exposure with the tall table's `imageid`, time, wavelength, measured flux, SPHEREx's zodi column, ZodiPy, **our model at
that wavelength**, our model and ZodiPy at 5 μm, and the time to the nearest simulated exposure. This is the mentor's
"replace the ZodiPy column with ours" for these patches. Over all patches, ours − ZodiPy ≤ 0.06% (median per patch).


In [ ]:
T, h = readdlm(joinpath(D, "ttbin_L6_g1_000_ours.csv"), ',', header=true)
println(join(h, "  ")); for i in 1:5; println(join(T[i, :], "  ")); end
@printf("ours vs ZodiPy at the exposure wavelength, patch g1_000: max |Δ| %.3f%%\n", maximum(abs.(100 .* (T[:, 7] .- T[:, 6]) ./ T[:, 6])))


## 5. Caveats and what runs overnight
* No star masking (the user's rule for now). Medians over 2–5 million pixels per exposure resist single stars, but not
  the smooth sum of faint stars; that sum is the csc|b| term.
* Only 34 patches are off the plane, and the zodi brightness and |β| are strongly tied across them. So c1 says "scales
  with the zodi", not which part of the cloud is too faint.
* The time match uses the nearest simulated exposure (≤ 1 min), and the observer is Earth for ZodiPy, as before.
* **Overnight (Discovery job 695791):** the same at grid 2, i.e. 192 patches ~15° apart (228 partitions, 389 GB), all 8
  windows. That gives 4× the patches for the c1 spectrum, enough to map the offset ratio across the sky, and a check on
  the 3.45 μm point.
